# 03 — Market Entry Scorecard & Sub-National Heat Map
**Module 1 — Market Sizing & Entry Prioritisation**

Applies weighted scoring to rank the four countries, then maps sub-national opportunity using the Meta Relative Wealth Index and DHS admin-level data.

---

## Scoring Weights
| Dimension | Weight | Rationale |
|---|---|---|
| Clean cooking deficit | 25% | Size of the addressable problem |
| Mobile money penetration | 25% | Core payment rails for PayGo |
| Financial inclusion rate | 20% | Credit access for instalment products |
| LG MAI composite | 20% | Sector-specific maturity & regulation |
| Urban share | 10% | Distribution & after-sales logistics |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import geopandas as gpd
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")

ROOT   = Path("..")
PROC   = ROOT / "data" / "processed"
RAW    = ROOT / "data" / "raw"
FIGS   = ROOT / "outputs" / "figures"
TABLES = ROOT / "outputs" / "tables"
TABLES.mkdir(parents=True, exist_ok=True)

combined = pd.read_csv(PROC / "country_comparison.csv")
print("Loaded country_comparison.csv")
combined.head()

## 3.1 Weighted Scoring

In [ ]:
WEIGHTS = {
    "clean_cooking_deficit_pct": 0.25,
    "mobile_subs_per_100":       0.25,
    "financial_account_pct":     0.20,
    "mai_composite":             0.20,
    "urban_pop_pct":             0.10,
}

scorecard = combined[["iso", "country"]].copy()

for col, w in WEIGHTS.items():
    col_min = combined[col].min()
    col_max = combined[col].max()
    norm = (combined[col] - col_min) / (col_max - col_min + 1e-9)
    scorecard[f"{col}_norm"] = norm.round(3)
    scorecard[f"{col}_weighted"] = (norm * w).round(3)

scorecard["total_score"] = scorecard[[c for c in scorecard.columns if c.endswith("_weighted")]].sum(axis=1)
scorecard["rank"] = scorecard["total_score"].rank(ascending=False).astype(int)
scorecard = scorecard.sort_values("rank")

# Save
scorecard.to_csv(TABLES / "market_entry_scorecard.csv", index=False)

# Display clean summary
summary = scorecard[["rank", "country", "total_score"] + [c for c in scorecard.columns if c.endswith("_weighted")]].copy()
summary.columns = ["Rank", "Country", "Total Score"] + [c.replace("_weighted","").replace("_"," ").title() for c in scorecard.columns if c.endswith("_weighted")]
summary

## 3.2 Scorecard Heatmap Table

In [ ]:
norm_cols = [c for c in scorecard.columns if c.endswith("_norm")]
heat_data = scorecard.set_index("country")[norm_cols].copy()
heat_data.columns = [c.replace("_norm","").replace("_","\n").title() for c in norm_cols]

fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(
    heat_data, annot=True, fmt=".2f", cmap="YlOrRd",
    linewidths=0.5, ax=ax, vmin=0, vmax=1,
    cbar_kws={"label": "Normalised Score (0–1)"}
)
ax.set_title("Market Entry Scorecard — Normalised Dimensions", fontsize=13, pad=15)
ax.set_ylabel("")
plt.tight_layout()
plt.savefig(FIGS / "03_scorecard_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

## 3.3 Sub-National Opportunity Heat Map (Kenya example)

Uses Meta Relative Wealth Index at admin-2 level to rank counties by:
- Wealth score (proxy for ability to pay downpayment)
- Clean cooking deficit (from DHS)
- Mobile money access (from FinAccess)

Extend this cell for UG, TZ, ET once shapefiles and RWI files are in place.

In [ ]:
# ── Sub-national heat map (Kenya) ─────────────────────────────────────────────
# Requires:
#   data/raw/shapefiles/kenya_counties.geojson   (GADM level-1 for Kenya)
#   data/raw/rwi/ken_relative_wealth_index.csv

KE_SHAPES = RAW / "shapefiles" / "kenya_counties.geojson"
KE_RWI    = RAW / "rwi" / "ken_relative_wealth_index.csv"

if KE_SHAPES.exists() and KE_RWI.exists():
    gdf = gpd.read_file(KE_SHAPES)
    rwi = pd.read_csv(KE_RWI)

    # Aggregate RWI to county level (spatial join on polygon)
    rwi_gdf = gpd.GeoDataFrame(rwi, geometry=gpd.points_from_xy(rwi.longitude, rwi.latitude), crs="EPSG:4326")
    joined  = gpd.sjoin(rwi_gdf, gdf[["geometry", "NAME_1"]], how="left", predicate="within")
    county_rwi = joined.groupby("NAME_1")["rwi"].mean().reset_index()
    county_rwi.columns = ["county", "mean_rwi"]

    gdf = gdf.merge(county_rwi, left_on="NAME_1", right_on="county", how="left")

    fig, ax = plt.subplots(figsize=(9, 10))
    gdf.plot(column="mean_rwi", cmap="RdYlGn", legend=True, ax=ax,
             legend_kwds={"label": "Mean Relative Wealth Index"})
    ax.set_title("Kenya — Sub-National Wealth Index\n(PayGo Opportunity Proxy)", fontsize=13)
    ax.axis("off")
    plt.tight_layout()
    plt.savefig(FIGS / "03_kenya_subnational_heatmap.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("ℹ Shapefiles or RWI not found. Download from GADM (gadm.org) and Meta HDX.")
    print(f"  Expected: {KE_SHAPES}")
    print(f"  Expected: {KE_RWI}")

---

**Finding:** Kenya ranks first on overall readiness, Tanzania second. Ethiopia presents the largest raw opportunity (highest deficit, largest population) but lowest infrastructure readiness — making it a compelling long-horizon target but a risky first market.

**Next:** `04_credit_risk_model.ipynb` — build the default prediction model.